In [1]:
import os
import gc
import json
import time
import shutil
import requests
import numpy as np
import pandas as pd
from PIL import Image
from dotenv import load_dotenv
from huggingface_hub import HfApi, create_repo, repo_exists

/home/trob/trob-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
extracted_folder_path = 'entities_dataset_v2/'
download_folder_path = 'images'
resize_folder_path = 'images/resize'  # مسیر ذخیره تصاویر resize شده

In [3]:
def check_file_exists_in_hf(filename, repo_id="alizali/torob_images_parquet_resize"):
    """
    Check if the file has already been uploaded to Hugging Face
    """
    load_dotenv()
    try:
        api = HfApi(token=os.getenv("HF_TOKEN"))

        if not repo_exists(repo_id=repo_id, repo_type="dataset"):
            create_repo(repo_id=repo_id, repo_type="dataset", private=False)
            print(f"create repo {repo_id}")
        else:
            print(f"exist repo {repo_id}")

        files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
        
        if filename in files:
            print(f"  File {filename} already uploaded. Skipping...")
            return True
        else:
            return False
            
    except Exception as e:
        print(f"  File check: {e}")
        return False

In [4]:
def sort_json_files_by_length(extracted_folder_path):
    """Sort JSON files by number of items"""
    json_files = [f for f in os.listdir(extracted_folder_path) if f.endswith('.json')]
    
    file_lengths = []
    for json_file in json_files:
        json_file_path = os.path.join(extracted_folder_path, json_file)
        try:
            with open(json_file_path, 'r') as f:
                data = json.load(f)
                file_lengths.append((json_file, len(data)))
        except Exception as e:
            print(f"Error reading {json_file}: {e}")
    
    sorted_files = sorted(file_lengths, key=lambda x: x[1])
    return sorted_files

In [5]:
def download_and_resize_images(json_file_path, resize_output_folder, target_size=(224, 224), max_retries=3):
    """
    Download images from URLs, resize them to target_size, and save to resize folder
    """
    os.makedirs(resize_output_folder, exist_ok=True)

    try:
        with open(json_file_path, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error loading JSON: {e}")
        return 0

    downloaded_count = 0
    failed_count = 0
    
    for entry in data:
        if 'image_url' in entry:
            image_url = entry['image_url']
            image_name = image_url.split('/')[-1]
            
            # Retry mechanism
            success = False
            for attempt in range(max_retries):
                try:
                    # Download image
                    resp = requests.get(
                        image_url, 
                        timeout=30,
                        headers={'User-Agent': 'Mozilla/5.0'},
                        allow_redirects=True
                    )
                    resp.raise_for_status()
                    
                    # Open image from bytes
                    from io import BytesIO
                    img = Image.open(BytesIO(resp.content)).convert('RGB')
                    
                    # Resize to target size
                    img_resized = img.resize(target_size, Image.LANCZOS)
                    
                    # Save resized image
                    save_path = os.path.join(resize_output_folder, image_name)
                    img_resized.save(save_path)
                    
                    downloaded_count += 1
                    success = True
                    
                    # Free memory
                    del img, img_resized
                    
                    if downloaded_count % 50 == 0:
                        print(f"Downloaded and resized: {downloaded_count} images")
                    break
                    
                except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                    if attempt < max_retries - 1:
                        print(f"Timeout on {image_name}, retry {attempt + 1}/{max_retries}...")
                        continue
                    else:
                        print(f"Failed after {max_retries} attempts: {image_name}")
                        failed_count += 1
                        
                except Exception as e:
                    print(f"Failed to download {image_name}: {e}")
                    failed_count += 1
                    break
            
            if not success:
                failed_count += 1
    
    print(f"Download complete: {downloaded_count} succeeded, {failed_count} failed")
    return downloaded_count

In [6]:
def images_to_parquet(input_folder, output_file):
    """
    Read all images in the input folder,
    convert each image to numpy array,
    and save all in one Parquet file.
    """
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    image_data = []

    # Iterate through all files in folder
    image_files = [f for f in os.listdir(input_folder) 
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp'))]
    
    total_images = len(image_files)
    
    if total_images == 0:
        print(f"No images found to convert to Parquet!")
        return False
    
    print(f"Converting {total_images} images to Parquet...")
    
    for idx, filename in enumerate(image_files, 1):
        file_path = os.path.join(input_folder, filename)
        try:
            # Open image and convert to array
            img = Image.open(file_path).convert('RGB')
            arr = np.array(img)

            # Convert array to bytes for DataFrame storage
            image_data.append({
                "filename": filename,
                "array": arr.tobytes(),
                # "shape": arr.shape
            })
            
            # Free memory after each image
            del img, arr
            
            if idx % 100 == 0:
                print(f"  Processed: {idx}/{total_images}")
                gc.collect()
            
        except Exception as e:
            print(f"Error reading {filename}: {e}")

    if not image_data:
        print(f"No images found to convert to Parquet!")
        return False

    # Create DataFrame from data
    print(f"Creating DataFrame with {len(image_data)} images...")
    df = pd.DataFrame(image_data)

    # Save to Parquet format
    print(f"Saving to Parquet...")
    df.to_parquet(output_file, index=False, compression='snappy')
    print(f"File {output_file} saved successfully.")
    
    # Aggressive memory cleanup
    del df
    del image_data
    gc.collect()
    gc.collect()
    
    return True

In [7]:
def upload_to_huggingface(local_file_path, repo_id="alizali/torob_images_parquet_resize", max_retries=5):
    """
    Upload file to Hugging Face with retry mechanism
    """
    load_dotenv()
    
    for attempt in range(max_retries):
        try:
            api = HfApi(token=os.getenv("HF_TOKEN"))
            dest_path = os.path.basename(local_file_path)
            
            # Create repo if it doesn't exist
            try:
                api.create_repo(
                    repo_id=repo_id,
                    repo_type="dataset",
                    exist_ok=True,
                    private=False,
                )
            except Exception as e:
                print(f"Error creating or accessing repo: {e}")

            # Upload with timeout settings
            print(f"Uploading {dest_path} (attempt {attempt + 1}/{max_retries})...")
            api.upload_file(
                path_or_fileobj=local_file_path,
                path_in_repo=dest_path,
                repo_id=repo_id,
                repo_type="dataset",
            )
            print(f"File {dest_path} uploaded to repo {repo_id}.")
            return True
            
        except Exception as e:
            error_msg = str(e)
            print(f"Upload error (attempt {attempt + 1}/{max_retries}): {error_msg}")
            
            # Check if it's a server error (500)
            if "500" in error_msg or "Internal Server Error" in error_msg:
                if attempt < max_retries - 1:
                    wait_time = (attempt + 1) * 10
                    print(f"Server error 500, waiting {wait_time} seconds before retry...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"Failed after {max_retries} attempts due to server error")
                    return False
            else:
                print(f"Non-retryable error: {error_msg}")
                return False
    
    return False

In [8]:
def remove_local_folder(folder_path):
    """Remove local folder"""
    try:
        if os.path.exists(folder_path):
            shutil.rmtree(folder_path)
            print(f"Removed local folder: {folder_path}")
    except Exception as e:
        print(f"Failed to remove {folder_path}: {e}")

In [9]:
def remove_local_file(file_path):
    """Remove local file"""
    try:
        if os.path.exists(file_path):
            os.remove(file_path)
            print(f"Removed local file: {file_path}")
    except Exception as e:
        print(f"Failed to remove {file_path}: {e}")

In [10]:
def process_json_and_upload(extracted_folder_path, resize_folder_path):
    """Process each JSON file and upload to Hugging Face"""
    
    sorted_files = sort_json_files_by_length(extracted_folder_path)
    
    total_files = len(sorted_files)
    processed_count = 0
    skipped_count = 0

    # Process each JSON file
    for json_file, item_count in sorted_files:
        json_file_path = os.path.join(extracted_folder_path, json_file)

        # Folder name for storing resized images
        resize_output_folder = os.path.join(resize_folder_path, json_file.split('.')[0])

        # Create unique Parquet filename for each JSON file
        parquet_filename = f"{json_file.split('.')[0]}.parquet"
        parquet_output = os.path.join("output_parquet", parquet_filename)
        
        print(f"\n{'='*70}")
        print(f"Processing: {json_file} ({item_count} items)")
        print(f"{'='*70}")
        
        # Check if file has already been uploaded
        if check_file_exists_in_hf(parquet_filename):
            skipped_count += 1
            print(f"Skipping {json_file} - already uploaded ({skipped_count} skipped)")
            continue
        
        try:
            # Download and resize images
            print("Downloading and resizing images to 224x224...")
            downloaded = download_and_resize_images(
                json_file_path, 
                resize_output_folder,
                target_size=(224, 224)
            )
            
            if downloaded == 0:
                print(f"No images downloaded for {json_file}")
                continue

            # Convert resized images to Parquet
            print("Converting to Parquet...")
            if not images_to_parquet(resize_output_folder, parquet_output):
                print(f"Error converting {json_file} to Parquet")
                remove_local_folder(resize_output_folder)
                gc.collect()
                continue

            # Upload to Hugging Face
            print("Uploading to Hugging Face...")
            upload_success = upload_to_huggingface(parquet_output, repo_id="alizali/torob_images_parquet_resize")
            
            if not upload_success:
                print(f"Error uploading {json_file}")
                remove_local_folder(resize_output_folder)
                if os.path.exists(parquet_output):
                    remove_local_file(parquet_output)
                gc.collect()
                continue

            # Remove local resized images folder
            print("Cleaning up resized images folder...")
            remove_local_folder(resize_output_folder)
            
            # Remove local Parquet file
            print("Cleaning up Parquet file...")
            remove_local_file(parquet_output)
            
            # Aggressive RAM cleanup
            print("Freeing RAM...")
            gc.collect()
            gc.collect()
            gc.collect()
            
            processed_count += 1
            print(f"Success: {json_file} ({processed_count}/{total_files - skipped_count} processed)")
            print(f"RAM cleanup completed\n")
            
        except Exception as e:
            print(f"Unexpected error in {json_file}: {e}")
            # Cleanup on error
            if os.path.exists(resize_output_folder):
                remove_local_folder(resize_output_folder)
            if os.path.exists(parquet_output):
                remove_local_file(parquet_output)
            gc.collect()
            gc.collect()
            continue
    
    # Final report
    print(f"\n{'='*70}")
    print(f"Final Report:")
    print(f"  Processed: {processed_count}")
    print(f"  Skipped (already uploaded): {skipped_count}")
    print(f"  Total files: {total_files}")
    print(f"{'='*70}")

In [ ]:
process_json_and_upload(extracted_folder_path, resize_folder_path)


Processing: 5989_entities_dataset_v2.json (240 items)
exist repo alizali/torob_images_parquet_resize
  File 5989_entities_dataset_v2.parquet already uploaded. Skipping...
Skipping 5989_entities_dataset_v2.json - already uploaded (1 skipped)

Processing: 285_entities_dataset_v2.json (255 items)
exist repo alizali/torob_images_parquet_resize
  File 285_entities_dataset_v2.parquet already uploaded. Skipping...
Skipping 285_entities_dataset_v2.json - already uploaded (2 skipped)

Processing: 4984_entities_dataset_v2.json (255 items)
exist repo alizali/torob_images_parquet_resize
  File 4984_entities_dataset_v2.parquet already uploaded. Skipping...
Skipping 4984_entities_dataset_v2.json - already uploaded (3 skipped)

Processing: 2693_entities_dataset_v2.json (255 items)
exist repo alizali/torob_images_parquet_resize
  File 2693_entities_dataset_v2.parquet already uploaded. Skipping...
Skipping 2693_entities_dataset_v2.json - already uploaded (4 skipped)

Processing: 3283_entities_dataset_

Processing Files (1 / 1): 100%|██████████| 21.4MB / 21.4MB, 17.9MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.


File 2912_entities_dataset_v2.parquet uploaded to repo alizali/torob_images_parquet_resize.
Cleaning up resized images folder...
Removed local folder: images/resize/2912_entities_dataset_v2
Cleaning up Parquet file...
Removed local file: output_parquet/2912_entities_dataset_v2.parquet
Freeing RAM...
Success: 2912_entities_dataset_v2.json (1/101 processed)
RAM cleanup completed


Processing: 4667_entities_dataset_v2.json (256 items)
exist repo alizali/torob_images_parquet_resize
  File 4667_entities_dataset_v2.parquet already uploaded. Skipping...
Skipping 4667_entities_dataset_v2.json - already uploaded (158 skipped)

Processing: 4666_entities_dataset_v2.json (256 items)
exist repo alizali/torob_images_parquet_resize
  File 4666_entities_dataset_v2.parquet already uploaded. Skipping...
Skipping 4666_entities_dataset_v2.json - already uploaded (159 skipped)

Processing: 3332_entities_dataset_v2.json (256 items)
exist repo alizali/torob_images_parquet_resize
  File 3332_entities_dataset_

In [ ]:
# def process_single_json(json_file_path,
#                         resize_folder_path="images/resize",
#                         output_parquet_dir="output_parquet",
#                         target_size=(224, 224)):
#     """
#     پردازش فقط یک JSON:
#     - دانلود و ریسایز تصاویر با download_and_resize_images
#     - تبدیل همه‌ی تصاویر ریسایز شده به یک Parquet با images_to_parquet
#     - پاک‌سازی فولدر ریسایز در پایان
#     """
#     os.makedirs(resize_folder_path, exist_ok=True)
#     os.makedirs(output_parquet_dir, exist_ok=True)

#     # نام پایه فایل برای نام‌گذاری خروجی‌ها
#     base = os.path.splitext(os.path.basename(json_file_path))[0]
#     resize_out = os.path.join(resize_folder_path, base)
#     parquet_out = os.path.join(output_parquet_dir, f"{base}.parquet")

#     # اگر فولدر قبلی وجود دارد، پاک کن تا تمیز شروع کنیم
#     if os.path.exists(resize_out):
#         try:
#             shutil.rmtree(resize_out)
#         except Exception as e:
#             print(f"Warn: could not remove existing resize folder: {e}")
#     os.makedirs(resize_out, exist_ok=True)

#     print("=" * 70)
#     print(f"Processing single JSON: {json_file_path}")
#     print("=" * 70)

#     # 1) دانلود و ریسایز
#     print("Downloading & resizing images ...")
#     downloaded = download_and_resize_images(
#         json_file_path=json_file_path,
#         resize_output_folder=resize_out,
#         target_size=target_size
#     )

#     if downloaded == 0:
#         print("No images downloaded. Nothing to convert.")
#         # پاک‌سازی فولدر ریسایز
#         try:
#             shutil.rmtree(resize_out)
#         except Exception as e:
#             print(f"Warn: cleanup resize folder failed: {e}")
#         gc.collect()
#         return None  # یا False

#     # 2) تبدیل به پارکت (تک‌فایل)
#     print("Converting resized images to Parquet ...")
#     try:
#         ok = images_to_parquet(resize_out, parquet_out)
#     except Exception as e:
#         print(f"Parquet conversion error: {e}")
#         ok = False

#     # پاک‌سازی فولدر ریسایز
#     try:
#         shutil.rmtree(resize_out)
#         print("Cleaned up resized images folder.")
#     except Exception as e:
#         print(f"Warn: cleanup resize folder failed: {e}")

#     gc.collect()

#     if ok:
#         print(f"✅ Done. Parquet saved at: {parquet_out}")
#         return parquet_out
#     else:
#         print("❌ Failed to create Parquet.")
#         return None  # یا False


In [ ]:
# single_json = "entities_dataset_v2/538_entities_dataset_v2.json"
# process_single_json(single_json)